# 01 - Sanity check pierwszych próbek

**Cel:** zweryfikować, że pipeline produkuje biologicznie sensowne dane.

Notebook czyta dane z **gotowej macierzy** `data/processed/expression_matrix.parquet`
(wynik komendy `build-matrix`) i porównuje ekspresję markerów między próbkami.

**Co weryfikuję:**
1. Macierz ekspresji ma oczekiwaną strukturę (60 660 genów × N próbek)
2. Powiązanie plik → pacjent → typ tkanki (przez `gdc_sample_sheet`)
3. Geny housekeeping wyrażane stabilnie między próbkami
4. **Counts vs TPM — pułapka głębokości sekwencjonowania**
5. Markery LUAD pokazują różnicę Normal vs Tumor

## Setup

In [1]:
import sys
from pathlib import Path

NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR if (NB_DIR / "src").exists() else NB_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT))

import polars as pl
import plotly.express as px

from src.ingest.star_parser import parse_star_counts
from src.ingest.sample_sheet_parser import parse_sample_sheet
from src.ingest.file_naming import STAR_FILE_PATTERNS

print("Project root:", PROJECT_ROOT)
print("polars:", pl.__version__)

Project root: /Users/luka/luad-huba-clean
polars: 1.40.1


## Wczytanie gotowej macierzy ekspresji

In [2]:
matrix_path = PROJECT_ROOT / "data" / "processed" / "expression_matrix.parquet"
assert matrix_path.exists(), f"Brak macierzy: {matrix_path}. Odpal `build-matrix` najpierw."

matrix = pl.read_parquet(matrix_path)
sample_cols = [c for c in matrix.columns if c != "gene_id"]

print(f"Wymiary: {matrix.height:,} genów × {len(sample_cols)} próbek")
print(f"Próbki: {sample_cols}")
matrix.head(5)

Wymiary: 60,660 genów × 2 próbek
Próbki: ['TCGA-44-2655-11A', 'TCGA-86-8358-01A']


gene_id,TCGA-44-2655-11A,TCGA-86-8358-01A
str,i64,i64
"""ENSG00000000003.15""",2497,669
"""ENSG00000000005.6""",7,0
"""ENSG00000000419.13""",1525,889
"""ENSG00000000457.14""",803,442
"""ENSG00000000460.17""",165,662


## Mapowanie próbka → pacjent → typ tkanki

- kod `01A` → Primary Tumor
- kod `11A` → Solid Tissue Normal

In [4]:
sheet_path = next((PROJECT_ROOT / "data/raw").rglob("gdc_sample_sheet*.tsv"))
sheet = parse_sample_sheet(sheet_path)

sheet.select(["sample_id", "case_id", "tissue_type", "tcga_sample_code", "is_tumor"])

sample_id,case_id,tissue_type,tcga_sample_code,is_tumor
str,str,str,str,bool
"""TCGA-44-2655-11A""","""TCGA-44-2655""","""Normal""","""11A""",false
"""TCGA-86-8358-01A""","""TCGA-86-8358""","""Tumor""","""01A""",true


## Mapowanie gene_id → gene_name

Macierz trzyma identyfikatory Ensembl. Słownik wyciągam z jednego z plików STAR.
Dodatkowo zapamiętujem referencje do obu plików STAR — przydadzą się dalej
do porównania **counts vs TPM**.

In [7]:
raw_dir = PROJECT_ROOT / "data/raw"
star_files = []
for pattern in STAR_FILE_PATTERNS:
    star_files.extend(raw_dir.rglob(pattern))

if not star_files:
    raise FileNotFoundError(
        f"Nie znaleziono plików STAR-Counts w {raw_dir} (rekurencyjnie). "
        f"Wzorce: {STAR_FILE_PATTERNS}"
    )

star_files = sorted(star_files)
print(f"Znaleziono {len(star_files)} plik(ów) STAR")

reference = parse_star_counts(star_files[0]).select(["gene_id", "gene_name", "gene_type"])
gene_lookup = reference.select(["gene_name", "gene_id"]).to_dict(as_series=False)
name_to_id = dict(zip(gene_lookup["gene_name"], gene_lookup["gene_id"]))

star_by_stem = {f.name.split(".")[0].split("_")[0]: parse_star_counts(f) for f in star_files}
print(f"Załadowano {len(name_to_id):,} mapowań gene_name → gene_id")
print(f"Pliki STAR wczytane (dla porównania counts vs TPM): {list(star_by_stem.keys())}")

Znaleziono 2 plik(ów) STAR
Załadowano 59,427 mapowań gene_name → gene_id
Pliki STAR wczytane (dla porównania counts vs TPM): ['11d52676-7017-48b4-872a-cefb91b8651c', '461cd8f2-fe53-448d-b6a9-0a95792464c7']


## Sanity check 1: geny housekeeping (counts)

`ACTB`, `GAPDH`, `DPM1`, `TBP`, `HPRT1` — geny *housekeeping* kodują białka
niezbędne dla podstawowych funkcji każdej komórki (cytoszkielet, glikoliza,
glikozylacja, transkrypcja, recykling puryn). Założenie: ich poziom mRNA
powinien być stabilny między tkankami.

**Patrzymy na surowe `unstranded counts`**

In [8]:
HOUSEKEEPING = ["ACTB", "GAPDH", "DPM1", "TBP", "HPRT1"]

def gene_expression(matrix: pl.DataFrame, gene_names: list[str]) -> pl.DataFrame:
    ids = [name_to_id[name] for name in gene_names if name in name_to_id]
    sub = matrix.filter(pl.col("gene_id").is_in(ids))
    sub = sub.join(reference.select(["gene_id", "gene_name"]), on="gene_id")
    return sub.select(["gene_name"] + sample_cols).sort("gene_name")

gene_expression(matrix, HOUSEKEEPING)

gene_name,TCGA-44-2655-11A,TCGA-86-8358-01A
str,i64,i64
"""ACTB""",494558,145729
"""DPM1""",1525,889
"""GAPDH""",29892,41900
"""HPRT1""",1040,2619
"""TBP""",933,564


## Pułapka metodologiczna: counts vs TPM

Patrząc na surowe counts powyżej, wydaje się że **żaden** z genów housekeeping
nie jest "stabilny" — różnice są wielokrotne.

Surowe counts zależą od **dwóch** rzeczy:
1. Realnej ekspresji genu w próbce (to można pominać)
2. **Łącznej głębokości sekwencjonowania** próbki (artefakt — ile DNA poszło do sekwenatora)

Jeśli próbka A miała 70 mln odczytów, a B miała 28 mln — w A *każdy* gen
będzie miał ~2.5× więcej odczytów.

**TPM** (Transcripts Per Million) normalizuje przez tę głębokość — to *koncentracja*
mRNA, porównywalna między próbkami. Pliki z GDC mają tę wartość już policzoną
(`tpm_unstranded`).

In [10]:
depth_rows = [
    {"sample": sid, "total_counts": int(matrix[sid].sum())}
    for sid in sample_cols
]
depth_df = pl.DataFrame(depth_rows)
print("Łączna głębokość sekwencjonowania per próbka:")
print(depth_df)

ratio = depth_df["total_counts"][0] / depth_df["total_counts"][1]
print(f"\nStosunek głębokości {sample_cols[0]} / {sample_cols[1]}: {ratio:.2f}x")

Łączna głębokość sekwencjonowania per próbka:
shape: (2, 2)
┌──────────────────┬──────────────┐
│ sample           ┆ total_counts │
│ ---              ┆ ---          │
│ str              ┆ i64          │
╞══════════════════╪══════════════╡
│ TCGA-44-2655-11A ┆ 71766674     │
│ TCGA-86-8358-01A ┆ 28226833     │
└──────────────────┴──────────────┘

Stosunek głębokości TCGA-44-2655-11A / TCGA-86-8358-01A: 2.54x


In [11]:
def tpm_lookup(star_df: pl.DataFrame, gene_names: list[str]) -> dict[str, float]:
    sub = star_df.filter(pl.col("gene_name").is_in(gene_names))
    return dict(zip(sub["gene_name"].to_list(), sub["tpm_unstranded"].to_list()))

stems = list(star_by_stem.keys())
tpm_1 = tpm_lookup(star_by_stem[stems[0]], HOUSEKEEPING)
tpm_2 = tpm_lookup(star_by_stem[stems[1]], HOUSEKEEPING)

counts_data = gene_expression(matrix, HOUSEKEEPING)
counts_map_1 = dict(zip(counts_data["gene_name"].to_list(), counts_data[sample_cols[0]].to_list()))
counts_map_2 = dict(zip(counts_data["gene_name"].to_list(), counts_data[sample_cols[1]].to_list()))

rows = []
for gene in HOUSEKEEPING:
    c1, c2 = counts_map_1[gene], counts_map_2[gene]
    t1, t2 = tpm_1[gene], tpm_2[gene]
    rows.append({
        "gen": gene,
        f"counts_{sample_cols[0][-3:]}": c1,
        f"counts_{sample_cols[1][-3:]}": c2,
        "ratio_counts": round(c1 / c2, 2),
        f"TPM_{sample_cols[0][-3:]}": round(t1, 1),
        f"TPM_{sample_cols[1][-3:]}": round(t2, 1),
        "ratio_TPM": round(t1 / t2, 2),
    })

pl.DataFrame(rows)

gen,counts_11A,counts_01A,ratio_counts,TPM_11A,TPM_01A,ratio_TPM
str,i64,i64,f64,f64,f64,f64
"""ACTB""",494558,145729,3.39,5646.5,4212.2,1.34
"""GAPDH""",29892,41900,0.71,627.4,2226.6,0.28
"""DPM1""",1525,889,1.72,63.5,93.8,0.68
"""TBP""",933,564,1.65,20.2,31.0,0.65
"""HPRT1""",1040,2619,0.4,32.4,206.9,0.16


**Co tu widać:**

- **ACTB**: stosunek counts 3.4× → po normalizacji TPM ~1.3× → **prawie idealna
  stabilność**.
- **GAPDH, HPRT1, TBP, DPM1**: po normalizacji TPM dalej różnią się rzędowo.
  To nie błąd a **realna różnica biologiczna** między zdrową tkanką płuca
  a guzem.

**Wniosek metodologiczny:** lista "housekeeping" to *przybliżenie*. 
Pojedyncze geny referencyjne potrafią się różnić między tkankami,
fazami choroby, fazami cyklu komórkowego. Współczesne metody normalizacji
(`DESeq2`, `edgeR`, TMM) używają **wszystkich** genów albo dużych zbiorów
stabilnych genów wybranych empirycznie z danych, nie sztywnej listy pięciu.

Dla pipeline'u to oznacza:
- Surowe counts są OK do baseline (Cox PH na pojedynczym genie)
- Do porównań między próbkami warto używać TPM albo log-CPM
- Do analizy różnicowej ekspresji — używać DESeq2/edgeR, a nie ręcznie liczonych stosunków

## Sanity check 2: markery LUAD i tkanki płuca

- **SFTPC** — sygnatura pneumocytów typu II. Normal: niezwykle wysoka, Tumor: drastycznie niska (klasyczny dowód utraty różnicowania komórkowego).
- **NAPSA** — marker różnicowania gruczołowego. Silny spadek ekspresji w guzie sugeruje słabsze zróżnicowanie nowotworu niż w typowym wczesnym LUAD.
- **EGFR, KRAS, TP53** — kluczowe geny w LUAD. Wyraźny wzrost mRNA wskazuje na nadekspresję EGFR. W przypadku KRAS (onkogen) i TP53 (supresor) poziomy ekspresji pozostają stabilne, ponieważ ich ewentualne mutacje zmieniają funkcję białek, a nie liczbę transkryptów mRNA.
- **TTF1** — czynnik transkrypcyjny (lineage survival oncogene). Wykazuje nadekspresję w guzie kompensującą utratę różnicowania.

In [12]:
LUAD_MARKERS = ["SFTPC", "NAPSA", "EGFR", "KRAS", "TP53", "TTF1"]
gene_expression(matrix, LUAD_MARKERS)

gene_name,TCGA-44-2655-11A,TCGA-86-8358-01A
str,i64,i64
"""EGFR""",9006,236
"""KRAS""",2517,2014
"""NAPSA""",49630,321
"""SFTPC""",1795955,2289
"""TP53""",3982,1587
"""TTF1""",1174,745


## Wniosek

1. Macierz ekspresji ma oczekiwane wymiary (60 660 genów = GENCODE v36)
2. Powiązanie plik → pacjent → typ tkanki działa
3. **Counts są mylące bez normalizacji** — głębokość sekwencjonowania
   wprowadza systematyczną różnicę między próbkami
4. **Listy housekeeping to przybliżenia** — niektóre geny faktycznie się
   różnią między tkankami nawet po normalizacji
5. SFTPC dramatycznie wyższy w Normal niż w Tumor